# 01 · 探索性数据分析（EDA）

**目标**：在建模之前，先把数据的"形状"摸清楚，并回答四个问题：

1. 四种需求模式（稳定 / 促销 / 季节 / 长尾）在数据里真的长得不一样吗？
2. 促销到底带来了多少增量？会不会有促销后的回落？
3. 星期几的效应有多强？为什么 `lag_7` 是必须的基线特征？
4. 断货把销量截断了多少？如果直接用 `units_sold` 当真实需求会错多远？

> 说明：`demand_latent`（真实需求）和 `sku_pattern`（模式标签）是**模拟数据独有**的，
> 真实电商数据里拿不到。这里用它们做"体检"，建模时绝不使用。

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# ===========================================================================
# 定位项目根目录（即包含 src/config.py 的那一层）
#
# 为什么需要这一段：Jupyter 内核的工作目录 = **启动 Jupyter 时的目录**，
# 和你在浏览器里打开的是哪个 ipynb 文件毫无关系。
# 典型翻车场景：
#     Jupyter 内核 cwd = C:/Users/<用户名>    （从主目录启动的服务）
#     项目却在另一个盘，例如 D:/code/ai-ecommerce-demand-forecast-inventory
# 这时"从 cwd 向上找 src/config.py"永远找不到 —— 另一个盘不在 C:/Users 的祖先链上。
#
# 这里按成本从低到高做 4 路兜底：
#   ① PROJECT_ROOT_OVERRIDE 手动指定（默认留空，只在自动定位失败时才需要填）
#   ② 内核 cwd 逐级向上 + 检查每一级的一级子目录
#   ③ notebook 文件自身所在目录（VS Code / JupyterLab 会注入路径）
#   ④ 在几个常见根目录里按项目目录名扫描（限深度，不做全盘遍历）
# ===========================================================================
MARKER = Path("src") / "config.py"
PROJECT_DIRNAME = "ai-ecommerce-demand-forecast-inventory"

# ① 最优先。换电脑 / 挪了位置，只改这一行就够了（留空字符串则跳过这一步）
# ① 手动指定（默认留空 = 跳过，靠 ②③④ 自动定位）。
#    只有在自动定位失败、或想强制指定时才填，例如：
#    PROJECT_ROOT_OVERRIDE = r"D:/code/ai-ecommerce-demand-forecast-inventory"
PROJECT_ROOT_OVERRIDE = ""

_SKIP_DIRS = {".git", ".venv", "venv", "__pycache__", "node_modules", ".ipynb_checkpoints",
              "Windows", "Program Files", "Program Files (x86)", "AppData", "$Recycle.Bin",
              "System Volume Information", "anaconda3", "Anaconda3"}


def _find_up(start: Path, max_up: int = 6):
    """从 start 逐级向上找，并检查每一级的一级子目录（覆盖"在项目上一级启动"）"""
    for base in [start, *list(start.parents)[:max_up]]:
        if (base / MARKER).exists():
            return base
        try:
            for child in sorted(base.iterdir()):
                if child.is_dir() and child.name not in _SKIP_DIRS \
                        and not child.name.startswith((".", "_")) and (child / MARKER).exists():
                    return child
        except (OSError, PermissionError):
            continue
    return None


def _scan_common_roots(max_depth: int = 4):
    """在常见根目录里按项目目录名找，限深度，避免全盘遍历"""
    home = Path.home()
    seeds = [home, home / "Desktop", home / "Documents", home / "Downloads",
             Path("C:/"), Path("D:/"), Path("E:/"), Path("F:/"), Path("G:/")]
    seen = set()
    for seed in seeds:
        if not seed.exists():
            continue
        stack = [(seed, 0)]
        while stack:
            cur, depth = stack.pop()
            if cur in seen:
                continue
            seen.add(cur)
            if cur.name == PROJECT_DIRNAME and (cur / MARKER).exists():
                return cur
            if depth >= max_depth:
                continue
            try:
                for child in cur.iterdir():
                    if child.is_dir() and child.name not in _SKIP_DIRS \
                            and not child.name.startswith((".", "_", "$")):
                        stack.append((child, depth + 1))
            except (OSError, PermissionError):
                continue
    return None


def find_project_root():
    if PROJECT_ROOT_OVERRIDE and (Path(PROJECT_ROOT_OVERRIDE) / MARKER).exists():
        return Path(PROJECT_ROOT_OVERRIDE).resolve(), "① 硬编码绝对路径"

    root = _find_up(Path.cwd().resolve())
    if root:
        return root.resolve(), "② 内核工作目录向上查找"

    nb_file = globals().get("__vsc_ipynb_file__") or os.environ.get("JPY_SESSION_NAME")
    if nb_file:
        root = _find_up(Path(nb_file).resolve().parent)
        if root:
            return root.resolve(), "③ notebook 文件所在目录"

    root = _scan_common_roots()
    if root:
        return root.resolve(), "④ 常见目录扫描"
    return None, None


ROOT, HOW = find_project_root()
if ROOT is None:
    raise RuntimeError(
        "找不到项目根目录（应包含 src/config.py）。\n"
        f"当前内核工作目录：{Path.cwd()}\n"
        "解决办法任选其一：\n"
        "  1) 把本单元的 PROJECT_ROOT_OVERRIDE 改成项目的绝对路径，重新运行本单元\n"
        "  2) 在项目根目录启动 Jupyter：cd ai-ecommerce-demand-forecast-inventory && jupyter notebook\n"
        "  3) 用项目自带的环境启动：双击项目根目录下的 start_jupyter.bat"
    )

# 切到项目根：让 notebook 里的相对路径与命令行完全一致
if Path.cwd().resolve() != ROOT:
    os.chdir(ROOT)

# insert(0) 而不是 append，避免撞上别的同名 src 包
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config

print(f"项目根目录：{ROOT}")
print(f"定位方式  ：{HOW}")
print(f"内核 Python：{sys.executable}")
print(f"依赖版本  ：pandas {pd.__version__} | numpy {np.__version__} | matplotlib {plt.matplotlib.__version__}")

# 主模型需要 lightgbm / xgboost 之一；本 notebook 的 EDA 不需要它们
_missing = [p for p in ("lightgbm", "xgboost") if importlib.util.find_spec(p) is None]
if len(_missing) == 2:
    print(f"提示：当前内核未安装 lightgbm / xgboost。EDA 不受影响，"
          f"但跑 src/train.py 时会自动降级到 sklearn 的 GradientBoosting（更慢、效果略差）。")

# 中文字体（缺字体时自动退化为默认字体，不影响运行）
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

In [ ]:
# 数据不存在时自动补跑前置脚本，保证 notebook 可以一键执行
if not config.CLEAN_FILE.exists():
    from src import data_prep
    data_prep.run()
if not config.FEATURE_FILE.exists():
    from src import features as feat_mod
    feat_mod.run()

clean = pd.read_csv(config.CLEAN_FILE, encoding="utf-8-sig", parse_dates=["date"])
feats = pd.read_csv(config.FEATURE_FILE, encoding="utf-8-sig", parse_dates=["date"])

print(f"原始明细: {clean.shape[0]:,} 行 × {clean.shape[1]} 列 | "
      f"{clean['sku_id'].nunique()} 个 SKU | {clean['date'].nunique()} 天")
print(f"时间范围: {clean['date'].min():%Y-%m-%d} ~ {clean['date'].max():%Y-%m-%d}")
clean.head()

## 1. 数据概览与质量检查

先看三件事：日期是否连续、有没有重复、各字段的分布量级。

In [ ]:
# 日期连续性：每个 SKU 都应该有相同的天数，否则 lag/rolling 会错位
days_per_sku = clean.groupby("sku_id")["date"].count()
print("每个 SKU 的天数取值:", sorted(days_per_sku.unique()))
print("重复的 (sku, date) 行数:", int(clean.duplicated(['sku_id', 'date']).sum()))
print("负销量行数:", int((clean['units_sold'] < 0).sum()))
print()
clean[["price", "units_sold", "on_hand_inventory", "lead_time_days"]].describe().round(2)

In [ ]:
# 四种需求模式的量级差异极大：长尾 SKU 日均不到 3 件，稳定型上百件
profile = (clean.groupby("sku_pattern")
           .agg(SKU数=("sku_id", "nunique"),
                日均销量=("units_sold", "mean"),
                日均真实需求=("demand_latent", "mean"),
                销量标准差=("units_sold", "std"),
                变异系数CV=("units_sold", lambda s: s.std() / max(s.mean(), 1e-9)),
                零销量占比=("units_sold", lambda s: (s == 0).mean()),
                断货率=("stockout_flag", "mean"),
                促销日占比=("promo_flag", "mean"))
           .round(3)
           .loc[list(config.PATTERNS)])
profile

**读法**：变异系数 CV = 标准差 / 均值。稳定型 CV 最低，长尾型最高而且零销量占比极高——
这决定了两类 SKU 不能用同一套误差假设（`inventory.py` 里的正态假设对长尾 SKU 是有偏的，README 已列为局限）。

## 2. 四种模式的时间形态

按模式把同一模式内所有 SKU 的日均销量聚合成一条曲线，看形态差异。

In [ ]:
fig, axes = plt.subplots(len(config.PATTERNS), 1, figsize=(11, 2.1 * len(config.PATTERNS)), sharex=True)
for ax, pattern in zip(axes, config.PATTERNS):
    sub = clean[clean["sku_pattern"] == pattern]
    daily = sub.groupby("date")["units_sold"].mean()
    promo = sub.groupby("date")["promo_flag"].max()
    ax.plot(daily.index, daily.values, color="#2c7fb8", lw=1.3, label="日均销量（观测）")
    ax.fill_between(daily.index, 0, daily.values.max(),
                    where=promo.reindex(daily.index).fillna(0).astype(bool),
                    color="#e4572e", alpha=0.12, label="促销日")
    ax.set_title(f"{pattern}（{sub['sku_id'].nunique()} 个 SKU）", fontsize=10, loc="left")
    ax.set_ylabel("件/天", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7, frameon=False, loc="upper left")
axes[-1].set_xlabel("日期", fontsize=8)
fig.suptitle("四种需求模式的日均销量形态", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

- **stable**：地平线平稳，只有微弱波动 → 移动平均就能打到很准；
- **promo**：平时低水位，促销瞬间拉到 3~5 倍 → 纯时序模型会系统性低估尖峰，必须给促销特征；
- **seasonal**：明显的中周期起伏 + 强周末效应 → 需要日历和滑动窗口同时刻画；
- **longtail**：大量接近 0 的日子 + 偶发爆量 → 误差分布长尾，用 MAE 比 RMSE 更合适。

## 3. 促销弹性：促销真的带来增量吗？有回落吗？

计算每个模式的"促销日 / 非促销日"销量倍数，并检查促销结束后 1~2 天的销量。
（剔除断货行，否则促销把库存卖空会污染结论）

In [ ]:
d = clean[clean["stockout_flag"] == 0]
lift = (d.groupby(["sku_pattern", "promo_flag"])["units_sold"].mean()
        .unstack("promo_flag").rename(columns={0: "非促销日均", 1: "促销日均"}))
lift["促销倍数"] = (lift["促销日均"] / lift["非促销日均"]).round(2)
print("各模式的促销倍数（剔除断货行）")
print(lift.round(2).loc[list(config.PATTERNS)].to_string())

# 促销结束后的回落：以"促销最后一天"为 0 天，看 t+1 / t+2 相对基准的水位
rows = []
for sku, g in d.groupby("sku_id"):
    g = g.sort_values("date").reset_index(drop=True)
    promo = g["promo_flag"].to_numpy()
    base = g.loc[promo == 0, "units_sold"].mean()
    ends = [i for i in range(len(g) - 1) if promo[i] == 1 and promo[i + 1] == 0]
    for offset in (1, 2, 3):
        vals = [g.loc[i + offset, "units_sold"] / base for i in ends
                if i + offset < len(g) and base > 0]
        if vals:
            rows.append({"sku_pattern": g["sku_pattern"].iloc[0], "t_plus": offset,
                         "相对基准水位": np.mean(vals)})
post = pd.DataFrame(rows).groupby(["sku_pattern", "t_plus"])["相对基准水位"].mean().unstack()
print("\n促销结束后的相对水位（1.00 = 回到日常水平）")
print(post.round(2).loc[list(config.PATTERNS)].to_string())

**要点**：促销日销量是日常的 2.5~4 倍（促销型 SKU 最高，3.96 倍），促销结束后第 1 天水位掉到 0.78~0.82、第 2 天回到 0.86~0.94 —— 这是典型的**提前购买（pull-forward）**。
如果模型只看当天的 `promo_flag`，会把这部分回落当成"需求突然变差"，所以 `features.py` 里额外加了 `promo_days_last_7`，让模型知道"前几天刚促过"。

## 4. 周内效应：为什么 `lag_7` 是必须的

看每个模式在周一~周日的相对销量水平（以各 SKU 自身均值为 1）。

In [ ]:
d = clean[clean["stockout_flag"] == 0].copy()
d["dow"] = d["date"].dt.day_of_week
# 先按 SKU 去均值，再按模式汇总，避免大 SKU 拉偏结果
d["idx"] = d["units_sold"] / d.groupby("sku_id")["units_sold"].transform("mean")
dow = (d.groupby(["sku_pattern", "dow"])["idx"].mean().unstack()
       .loc[list(config.PATTERNS)])
dow.columns = ["周一", "周二", "周三", "周四", "周五", "周六", "周日"]
print("周内指数（1.00 = 该 SKU 的日常水平）")
print(dow.round(2).to_string())

ax = dow.T.plot(kind="bar", figsize=(9, 3.2), width=0.78,
                color=["#2c7fb8", "#e4572e", "#c9b23f", "#7b2cbf"])
ax.axhline(1.0, color="grey", lw=0.8, ls="--")
ax.set_ylabel("相对销量指数")
ax.set_title("周内效应：季节型 SKU 周末比工作日高 30% 以上", fontsize=10, loc="left")
ax.legend(fontsize=8, frameon=False, ncol=4)
plt.xticks(rotation=0)
plt.show()

季节型的周末效应远强于其他模式：周一 0.79、周六 1.30，**周末比周初高约 60%**。这意味着：

- 用"最近 3 天均值"做预测会在周中高估、周末低估 → 滑动窗口至少要覆盖一个完整周（`rolling_7`）；
- `lag_7`（上周同一天）天然携带了周内位置信息，所以它是很难打败的基线；
- 长尾型几乎没有周内规律（0.91~1.10），再次说明它和其他三类不该共用一套假设。

## 5. 断货截断：观测销量低估了多少需求

这是整个项目最容易被忽略、也最影响结论的一步。

In [ ]:
so = clean[clean["stockout_flag"] == 1]
print(f"断货行数: {len(so):,} / {len(clean):,} = {len(so)/len(clean):.2%}")
print(f"其中完全卖空（units_sold = 0）的行数: {int((so['units_sold'] == 0).sum()):,}")
print()
cmp = (so.groupby("sku_pattern")
       .agg(断货天数=("date", "count"),
            观测销量均值=("units_sold", "mean"),
            真实需求均值=("demand_latent", "mean")))
cmp["需求被低估比例"] = (1 - cmp["观测销量均值"] / cmp["真实需求均值"]).round(3)
print("断货日的观测销量 vs 真实需求（真实需求只有模拟数据才有）")
print(cmp.round(3).loc[list(config.PATTERNS)].to_string())

# 整体口径：如果直接拿 units_sold 当需求，会低估多少总需求？
gap = 1 - clean["units_sold"].sum() / clean["demand_latent"].sum()
print(f"\n全量口径：观测总销量比真实总需求低 {gap:.2%}")

**结论（数据层面）**：断货日的观测销量平均只有真实需求的 30%~47%（稳定型被低估 70%、促销型被低估 53%），
全量口径观测总销量比真实总需求低 **5.07%**。直接拿 `units_sold` 学习，就等于把"缺货"学成"需求低"。

所以 `train.py` 在训练/验证时剔除了 `stockout_flag=1` 的行，`evaluate.py` 的主指标也只算未断货行。
注意：这只是一个保守的补丁，真正做生产时应该做截断需求还原（见 README 局限一节）。

## 6. 自相关：滞后特征该取几天

对每个 SKU 计算 `corr(y_t, y_{t-k})`（k=1..14），再按模式平均。

In [ ]:
def acf_by_sku(g, max_lag=14):
    """按 SKU 计算 corr(y_t, y_{t-k})。
    用完整观测序列、不删断货行：一旦删行，lag k 就不再是"k 天前"，相关系数会失真。"""
    y = g.sort_values("date")["units_sold"].to_numpy(dtype=float)
    out = {}
    for k in range(1, max_lag + 1):
        if len(y) > k + 10 and y[:-k].std() > 0 and y[k:].std() > 0:
            out[k] = float(np.corrcoef(y[:-k], y[k:])[0, 1])
    return pd.Series(out)


# 用显式循环，而不是 groupby.apply(include_groups=False)：
# include_groups 是 pandas 2.2 才引入的参数，pandas 2.0 / 1.x 会直接 TypeError
# （本机 Anaconda 是 pandas 2.0.3，就会炸）。显式循环在 1.x / 2.0 / 2.2 / 3.x 上行为一致。
_acf_rows = {}
for (_pattern, _sku), _g in clean.groupby(["sku_pattern", "sku_id"]):
    _acf_rows[(_pattern, _sku)] = acf_by_sku(_g)

acf = pd.DataFrame.from_dict(_acf_rows, orient="index").astype(float)
acf.index = pd.MultiIndex.from_tuples(acf.index, names=["sku_pattern", "sku_id"])
acf = acf.groupby(level=0).mean().loc[list(config.PATTERNS)]

print("按模式平均的自相关系数")
print(acf.round(3).to_string())

ax = acf.T.plot(figsize=(9, 3.0), marker="o", ms=3,
                color=["#2c7fb8", "#e4572e", "#c9b23f", "#7b2cbf"])
ax.set_xlabel("滞后天数 k")
ax.set_ylabel("自相关系数")
ax.set_title("自相关：k=1 主导，k=7 出现周内重复的次高点", fontsize=10, loc="left")
ax.legend(fontsize=8, frameon=False, ncol=4)
plt.show()

**读法**（数据实际长这样）：

- `k=1` 全面最高 —— 昨天是今天最好的单点预测；
- `k=7` 出现明显次高点，季节型 0.248、稳定型 0.086，而 `k=5/6` 反而更低 —— 这就是"上周同一天"有效的来源，它天然携带了周内位置信息；
- 促销型 `k=7` 只有 0.018，因为它的销量主要由促销驱动而不是由周内位置驱动；
- **长尾型所有滞后都在 0 附近**：日销量接近白噪声，历史值几乎不含信息。这不是模型没调好，而是这类商品的本质 —— 也直接解释了它后面 WMAPE 最差（80%+）。

## 7. 价格与销量：价格本身几乎没有解释力

直觉上价格越高销量越低，但在 SKU 内部、非促销日，这个关系可能非常弱。
先用相关系数验证一下，再讨论 `price_ratio` 为什么在特征重要性里排得很靠前。

In [ ]:
d = clean[(clean["promo_flag"] == 0) & (clean["stockout_flag"] == 0)]

# 同样避开 groupby.apply(include_groups=...) 的 pandas 版本差异
_price_corr = {}
for (_pattern, _sku), _g in d.groupby(["sku_pattern", "sku_id"]):
    _price_corr[(_pattern, _sku)] = float(_g["price"].corr(_g["units_sold"]))

corr = pd.Series(_price_corr, name="价格-销量相关系数").to_frame()
corr.index = pd.MultiIndex.from_tuples(corr.index, names=["sku_pattern", "sku_id"])
corr = corr.groupby(level=0).mean()
corr["绝对值的均值"] = corr["价格-销量相关系数"].abs()
print(corr.round(3).loc[list(config.PATTERNS)].to_string())

print("\n非促销日的价格波动只有 ±1.5% 的随机噪声，SKU 内部价格与销量几乎是零相关（|r| 在 0.01 量级）。")
print("真正带价格信息的是促销：促销日价格系数 0.80。所以 price_ratio 在模型里重要性偏高，")
print("本质是它在替 promo_flag 做交叉验证（折扣越深说明越是促销日），而不是价格弹性本身。")

## 8. 建模结果速览

跑完 `python -m src.train` 和 `python -m src.evaluate` 之后，这里可以直接看结果。

In [ ]:
if config.FORECAST_TEST_FILE.exists():
    fc = pd.read_csv(config.FORECAST_TEST_FILE, encoding="utf-8-sig", parse_dates=["date"])
    d = fc[fc["stockout_flag"] == 0]
    rows = []
    for name, label in [("baseline_last_week", "基线-上周同日"), ("baseline_ma_7", "基线-近7天均值"),
                        ("model", "主模型 GBDT")]:
        col = f"y_pred_{name}"
        err = (d[config.TARGET] - d[col]).abs()
        rows.append({"模型": label,
                     "MAE": err.mean(),
                     "RMSE": np.sqrt((err ** 2).mean()),
                     "WMAPE%": err.sum() / d[config.TARGET].sum() * 100})
    display(pd.DataFrame(rows).set_index("模型").round(3))
else:
    print("尚未生成预测结果，请先运行：python -m src.train && python -m src.evaluate")

In [ ]:
if config.FORECAST_TEST_FILE.exists():
    fc = pd.read_csv(config.FORECAST_TEST_FILE, encoding="utf-8-sig", parse_dates=["date"])
    # 挑一个长尾 SKU 看预测曲线（长尾是最难的一类）
    sku = fc[fc["sku_pattern"] == "longtail"]["sku_id"].iloc[0]
    g = fc[fc["sku_id"] == sku].sort_values("date")
    fig, ax = plt.subplots(figsize=(10, 2.8))
    ax.plot(g["date"], g[config.TARGET], color="#2c7fb8", lw=1.5, label="实际销量")
    ax.plot(g["date"], g["y_pred_model"], color="#e4572e", lw=1.4, label="主模型预测")
    ax.plot(g["date"], g["y_pred_baseline_ma_7"], color="#9aa5b1", lw=1.2, ls="--", label="基线 MA7")
    ax.scatter(g[g["stockout_flag"] == 1]["date"], g[g["stockout_flag"] == 1][config.TARGET],
               color="#7b2cbf", s=20, zorder=5, label="断货日")
    ax.set_title(f"测试期预测对比 · {sku}（longtail）", fontsize=10, loc="left")
    ax.set_ylabel("件/天", fontsize=8)
    ax.legend(fontsize=8, frameon=False, ncol=4)
    plt.show()

## 9. 小结（只写数据能支持的话）

1. 四种模式的可预测性差别极大：稳定型 MA7 已经能做到 16% WMAPE，促销型和长尾型才是模型真正发挥作用的地方；
2. 促销是外生冲击，促销倍数 2.5~4 倍、且促销后有 2 天的回落，`promo_flag` + `promo_days_last_7` 必须进模型；
3. 周内效应让 `lag_7` 成为强基线，任何模型的增量都应该以"打败 MA7"为门槛；
4. 断货截断让观测销量系统性低估需求（整体 5.07%，断货日高达 30%~47%），必须显式处理，
   否则安全库存会被算低，反而加剧断货；
5. 长尾 SKU 的自相关接近 0，说明它的问题不是"模型不够强"，而是信息本身不足 ——
   这类商品更应该走"分位数 / 间歇需求"专门方法，而不是硬套同一个 GBDT。

**下一步**：`src/train.py` 训练 → `src/evaluate.py` 评估 → `src/inventory.py` 转成补货决策。